Genomic prediction of Inflixmab in Inflammatory Bowel Disease Patients
This notebook uses baseline mucosal transcriptomic data to predict response to anti-TNF therapy in patients with active IBD refractory to corticosteroids and /or immunosuppression

In [1]:
import pandas as pd
import numpy as np
import sklearn


In [2]:
import pandas as pd
import numpy as np
import sklearn
import os
import urllib.request

# Define the URL for the GSE16879 Series Matrix file from NCBI GEO
data_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz"

# Define the local path where we want to save the file
dest_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# Check if the file already exists locally so we don't re-download it every time
if not os.path.exists(dest_path):
    print("Downloading dataset from NCBI GEO... This might take a minute.")
    urllib.request.urlretrieve(data_url, dest_path)
    print(f"Download complete! File saved to: {dest_path}")
else:
    print("Dataset already exists locally in the 'data' folder.")

Dataset already exists locally in the 'data' folder.


In [3]:
import gzip
import os

file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# 'rt' means read text. We add explicit encoding and tell it to ignore decoding errors
with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for i in range(40):
        line = f.readline().strip()
        # Show the first 120 characters of each line 
        print(f"Line {i+1}: {line[:120]}")

Line 1: !Series_title	"Mucosal expression profiling in patients with inflammatory bowel disease before and after first inflixima
Line 2: !Series_geo_accession	"GSE16879"
Line 3: !Series_status	"Public on Nov 03 2009"
Line 4: !Series_submission_date	"Jun 29 2009"
Line 5: !Series_last_update_date	"Mar 25 2019"
Line 6: !Series_pubmed_id	"19956723"
Line 7: !Series_summary	"We used microarrays to identify mucosal gene signatures predictive of response to infliximab (IFX) in p
Line 8: !Series_summary	""
Line 9: !Series_summary	"Keywords: drug response and treatment effect"
Line 10: !Series_overall_design	"Mucosal biopsies were obtained at endoscopy in actively inflamed mucosa from 61 IBD patients (24
Line 11: !Series_type	"Expression profiling by array"
Line 12: !Series_contributor	"Ingrid,,Arijs"
Line 13: !Series_contributor	"Leentje,,Van Lommel"
Line 14: !Series_contributor	"Kristel,,Van Steen"
Line 15: !Series_contributor	"Gert,,De Hertogh"
Line 16: !Series_contributor	"Karel,,Geboes"
Lin

In this section, we extract patient sample characteristics from the raw NCBI GEO metadata. Healthy controls are removed to isolate IBD (Chron's Diseaseand Ulcerative Colitis) patients and clean the strings to esetablish a Yes / No treatment response

In [4]:
metadata_dict = {}

# 1. Parse the file line-by-line to collect sample metadata
with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        # If we hit the expression matrix, stop reading metadata
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break # We reached the numeric data zone
                
        # Capture only the sample-specific metadata rows
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            key = parts[0]       # e.g., "!Sample_title"
            values = parts[1:]   # All the patient values for that row
            metadata_dict[key] = values

# 2. Convert our dictionary into a Pandas DataFrame
df_meta = pd.DataFrame.from_dict(metadata_dict)

# 3. Transpose the dataframe so patients are rows, and traits are columns
df_meta = df_meta.T

# 4. Clean up the index names for readability
df_meta.columns = [c.replace('"', '').strip() for c in df_meta.iloc[0]] # Clean column strings
# check first 5 rows and first 4 columns of table
df_meta.iloc[:5, :4]

,CO1,CO2,CO3,CO4
!Sample_title,"""CO1""","""CO2""","""CO3""","""CO4"""
!Sample_geo_accession,"""GSM364627""","""GSM364628""","""GSM364629""","""GSM364630"""
!Sample_status,"""Public on Nov 03 2009""","""Public on Nov 03 2009""","""Public on Nov 03 2009""","""Public on Nov 03 2009"""
!Sample_submission_date,"""Jan 26 2009""","""Jan 26 2009""","""Jan 26 2009""","""Jan 26 2009"""
!Sample_last_update_date,"""Aug 28 2018""","""Aug 28 2018""","""Aug 28 2018""","""Aug 28 2018"""


In [5]:
#transpose to make patients the rows, and the traits the collumns
df_clinical = df_meta.T 

#remove the exclamation mark and 'Sample_' prefix
df_clinical.columns = [col.replace('!Sample_', ' ').strip() for col in df_clinical.columns]

#read first 5 rows and 5 columns
df_clinical.iloc[:5, :5]

,title,geo_accession,status,submission_date,last_update_date
CO1,"""CO1""","""GSM364627""","""Public on Nov 03 2009""","""Jan 26 2009""","""Aug 28 2018"""
CO2,"""CO2""","""GSM364628""","""Public on Nov 03 2009""","""Jan 26 2009""","""Aug 28 2018"""
CO3,"""CO3""","""GSM364629""","""Public on Nov 03 2009""","""Jan 26 2009""","""Aug 28 2018"""
CO4,"""CO4""","""GSM364630""","""Public on Nov 03 2009""","""Jan 26 2009""","""Aug 28 2018"""
CO5,"""CO5""","""GSM364631""","""Public on Nov 03 2009""","""Jan 26 2009""","""Aug 28 2018"""


In [6]:
import gzip
import os
import pandas as pd

file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        # Stop reading when we hit the numeric data
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break 
                
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            
            # 1. Clean the key (e.g., "title")
            base_key = parts[0].replace('!Sample_', '').strip()
            
            # 2. Clean the values (the patient data list)
            values = [v.replace('"', '').strip() for v in parts[1:]]
            final_key = base_key # Start with the original name (e.g., "characteristics_ch1")
            
            counter = 1          # Start our number tag at 1

            # WHILE the name is already taken in the dictionary...
            while final_key in metadata_dict:
                # ...create a new name by gluing the counter to the original name
                final_key = f"{base_key}_{counter}"
                # ...and increase the counter by 1 for the next loop (if needed)
                counter +=1
            # ==========================================
            
            # 3. Save the data safely into the dictionary
            metadata_dict[final_key] = values

# Convert to DataFrame and Transpose
df_clinical = pd.DataFrame.from_dict(metadata_dict).T
list(df_clinical.columns)

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132]

In [7]:
# 1. Transpose the table 
df_clinical = df_clinical.T

# 2. Select the four characteristics columns, check top 5 rows
df_clinical[['characteristics_ch1', 'characteristics_ch1_1', 'characteristics_ch1_2', 'characteristics_ch1_3']].head(5)

,characteristics_ch1,characteristics_ch1_1,characteristics_ch1_2,characteristics_ch1_3
0,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
1,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
2,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
3,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
4,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...


In [8]:
#removing the controls from the IBD dataframe
df_ibd = df_clinical[ df_clinical['characteristics_ch1_2'] != "response to infliximab: Not applicable"]

#checking controls were removed
df_ibd['characteristics_ch1_2'].head(5)

6     response to infliximab: Yes
7     response to infliximab: Yes
8     response to infliximab: Yes
9     response to infliximab: Yes
10    response to infliximab: Yes
Name: characteristics_ch1_2, dtype: str

In [9]:
#renaming columns
df_ibd = df_ibd.rename(columns={'characteristics_ch1' : 'tissue'})
df_ibd = df_ibd.rename(columns={'characteristics_ch1_1' : 'disease'})
df_ibd = df_ibd.rename(columns={'characteristics_ch1_2' : 'response to infliximab'})
df_ibd = df_ibd.rename(columns={'characteristics_ch1_3' : 'before or after first infliximab treatment'})

In [10]:
#remove unnecessary text
df_ibd['tissue'] = df_ibd['tissue'].str.replace("tissue: ", "")
df_ibd['disease'] = df_ibd['disease'].str.replace("disease: ", "")
df_ibd['response to infliximab'] = df_ibd['response to infliximab'].str.replace("response to infliximab: ", "")
df_ibd['before or after first infliximab treatment'] = df_ibd['before or after first infliximab treatment'].str.replace("before or after first infliximab treatment: ", "")

df_ibd[['tissue', 'disease', 'response to infliximab', 'before or after first infliximab treatment']].head(5)

,tissue,disease,response to infliximab,before or after first infliximab treatment
6,Colon,UC,Yes,Before first infliximab treatment
7,Colon,UC,Yes,Before first infliximab treatment
8,Colon,UC,Yes,Before first infliximab treatment
9,Colon,UC,Yes,Before first infliximab treatment
10,Colon,UC,Yes,Before first infliximab treatment


We load the matrix containing 54,675 genes and filter it to only include rows corresponding to IBD patients. To ensure an unbiased evaluation, 20% of data is paritioned into a "hidden" test set before training the model

In [11]:
df_genes = pd.read_csv(
    file_path , #file defined earler
    sep='\t',   # columns are separated by tabs
    compression=  'gzip', #unzips file
    comment = '!', #ignores clinical header
    index_col = 0 #Gene IDs become row names
)

In [12]:
# 1. Sets row names to be patient IDs, not numbers
df_ibd = df_ibd.set_index('geo_accession')

# 2. Reload the gene matrix fresh (just to be safe!)
df_genes = pd.read_csv(
    file_path, 
    sep='\t', 
    compression='gzip', 
    comment='!', 
    index_col=0
)

# 3. Transpose the gene matrix
df_genes = df_genes.T 

# 4. Filter the gene matrix - only include rows that match patient IDs
df_genes = df_genes.loc[df_ibd.index]

# 5. Check the final dimensions
print("Clinical shape:", df_ibd.shape)
print("Gene shape:", df_genes.shape)

Clinical shape: (121, 34)
Gene shape: (121, 54675)


In [13]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df_genes, #genes
    df_ibd['response to infliximab'], #what we are trying to predict
    test_size = 0.2, #20% hidden from model
    random_state = 2 #random shuffle is the same each time
)
print(x_train.shape) #output is the 54675 genes for 96 patients (80%), used to train the model
print(x_test.shape) #output is the 54675 genes for 25 patients (20%) that is hidden from the model, "final exam"
#y train = answers for x_train (whether patient reponds) for model training
#y test = answers for x_test (whether patient responds), used to evaulate model performance

(96, 54675)
(25, 54675)


We imolement a L1-penalised (Lasso) Logistic regression model. The lasso penalty reduces the weights of non-predictive genes to 0, selecting for the most predictive biomarkers 

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
   penalty = 'l1',        #uses lasso penalty - deletes genes that have no impact
   solver = 'liblinear',  # math engine needed for L1 penalty
   random_state = 2 #keeps randomisation consistent
)

model.fit(x_train, y_train) #trains model based on patient genes and response to infliximab (80% of data)
print("Training data analysis finished")

Training data analysis finished


In [15]:
predictions = model.predict(x_test) #model predicts responses from training data for remainining 20% of patients

from sklearn.metrics import accuracy_score
final_grade = accuracy_score(y_test, predictions) #y_test is patient responses from x_test data, compapres against model predictions

print(f"The model scored: {final_grade * 100: .2f}%") # prints prediction

The model scored:  72.00%


We pull the non-zero coefficients from the model to identify the genes that survived the lasso filter. These are then sorted to isolate the top 10 most influential biomarkers

In [16]:
df_results = pd.DataFrame({ #creates dataframe using geness and weights from model
    'genes' : x_train.columns,
    'weight' : model.coef_[0],
})
 
df_results = df_results[df_results['weight'] != 0.0] #keeps weights not equal to 0

print(f"Total surviving genes: {len(df_results)}")             #len counts number of rows in dataframe

Total surviving genes: 1245


In [17]:
top_10_genes = df_results.sort_values('weight', ascending = False).head(10) # sorts by top 10 genes with most impact, saves genes to variable

top_10_genes.to_csv("infliximab_top_10_biomarkers.csv", index = False) #exports it to folder, index = false stops row numbers being added to file

print("Exported successfully")

import joblib

joblib.dump(model, "infliximab_lasso_model.pkl")

Exported successfully


['infliximab_lasso_model.pkl']